# 14 - Train Offline DQN with a Value-Gap Trace Cut

This notebook follows the same offline workflow as `02_train_offline_dqn.ipynb`. The
backup is still `DqnObjective`. The continuation is
`gate=value_gap_gate(beta=BETA, normalize=True, policy_delayed=False, value_delayed=False, bias=BIAS, eps=EPS)`:
the trace is cut by how far the taken action sits below `Q(s, a*)`, divided by
`|max_a Q| + eps` on the current head at that state. `a*` and the taken action are both
read from that head. `bias` is subtracted from the signed gap `Q(s, a_taken) - V`, then
the gap is capped at `0`.

At the state the action is taken from, `V = Q(s, a*)` and

`gap = min(0, (Q(s, a_taken) - V) / (|max_a Q| + eps) - bias)`

`c = 2 ** (gap / beta)` for positive beta; at `beta=0`, `c=1` for zero adjusted gap and
`c=0` for negative gap.

A gap of `0` continues with `c = 1`, the same as a `λ = 1` step. For positive beta, a
gap of `-beta` continues at `1/2`. A larger `beta` is a softer cut. When `bias` is `0`,
a tie with `max_a Q` on the current head is a gap of `0`. `normalize=True` divides by
`|max_a Q| + eps` on the value Q before `bias`. `normalize=False` turns that term off
and uses `gap = min(0, Q(s, a_taken) - V - bias)`, and takes no `eps`. A negative gap
shrinks `c` toward `0`, and the return bootstraps the delayed state value at that state.
The coefficient is a constant in the TD error. With `beta=0`, `bias=0`, and both delayed
flags `False`, `value_gap_gate` is the Watkins hard cut (`c` is `0` or `1`, and ties
continue). Combine it with `lambda_gate(td_lambda=...)` using
`general_gate(gates=(...))` to multiply the continuation by TD λ; this works for both
zero and positive beta.

The gate returns `[N, N]`. Row `t`, column `s` is the continuation at step `s` for the
return that started at `t` — the action taken from `s`, which the batch stores on the
next row. Entries with `s <= t` are unused. A zero discount ends the trace. A target
that still needs data beyond a group boundary raises.

Each event is tokenized by `Tokenizer` as text and embedded by the agent's separate
`Embedding` module.

- `type: "text"` — each field has its own `format=` and is tokenized as its own run.
  With `input_field=`, `format=` is an f-string and the step value is the name `field`
  (`{field+1}`, `{field:g}`). Omit `input_field=` for a const and set `format=` to the
  literal string. The word `field` is text there. `{field}` is still a placeholder and
  is rejected.
- `type: "token"` — literal `format` → one token ID. Nonempty output must encode to
  exactly one token. An optional `input_field` gates emission on presence. The
  action-request marker reuses the existing `\n` token without adding vocabulary
  entries.
- `type: "image"` — vision span (VL checkpoints; not used here)

There is no whole-step `format=`. Fields emit in `input_fields` order, each as a
separate HF tokenize call, so BPE never merges across fields. The coordinator's
`_episode_start` event emits `episode_prompt` as ordinary text before the first
observation of each outer episode. All inner episodes share that sequence and can attend
to the prompt. The producer and sampler own episode boundaries; the packer never inserts
a prompt. Incremental decode packs only new events. A new sample clears that stream's
context and calls `cache.reset_rows`. Collection omits zero rewards and false ending
flags. Absent or None input fields emit nothing. Collection records the zero-based
`repeat_episode_index` only on each inner reset event. The tokenizer renders it as a
one-based `episode N` header before that reset observation and inserts a blank line
after inner termination or truncation. These formatting newlines have no readout.
Missing fields stay absent in decoded views and optional token fields are skipped.
`_terminated` and `_truncated` are payload metadata mapped to the `terminated` and
`truncated` objective columns. `action_requested` emits the existing `\n` token with
`head_output=True`. Passive and terminal steps have no readout. Every input field
requires a specific provider name; optional inputs emit only when present.

This is a short usage example, not a full experiment. Evaluate a saved checkpoint in
`15_inference.ipynb`.


In [ ]:
import json
from copy import deepcopy
from functools import partial

import torch
from torch import nn

from mouse_core import AdamW, polyak_update
from mouse_core.data import (
    Augmenter,
    DataLoader,
    EnvironmentProtocol,
    Tokenizer,
    compose,
    environment_episode_ranges,
    load_stores_from_hub,
)
from mouse_core.models import (
    MLP,
    Backbone,
    DecodeCache,
    Embedding,
    Model,
    MultiEmbedding,
    prepare_decode,
    push_model_to_hub,
)
from mouse_core.objectives import DqnObjective, boundary_discount, value_gap_gate

# Repositories
# Hugging Face dataset repo for load_stores_from_hub
DATASET_ID = "mouse-example-frozenlake-dataset"
# Hugging Face model repo for push_model_to_hub
MODEL_ID = "mouse-example-model-value-gap-offline"

# Environment
# number of discrete actions predicted by the head
MAX_ACTIONS = 4
# vocabulary size for discrete observations
MAX_OBS_DISCRETE = 64
# max steps per episode (matches collect)
MAX_STEPS_PER_EPISODE = 24
# inner episodes per outer task (matches collect)
REPEAT_COUNT = 20

# Model
# HF checkpoint for Tokenizer, Embedding, and Backbone
PRETRAINED = "Qwen/Qwen3-0.6B"
HEAD_LAYERS = 1
# final prediction linear's initialization
HEAD_SCALE = 0.1

# Training
# packed tokens; whole episodes are added until the next one would not fit
TOKEN_BUDGET = 16384
# outer train cycles (print cadence)
NUM_CYCLES = 2
# optimizer updates per cycle (passed to run_train)
TRAIN_STEPS = 50

# Objective
# True divides by |max_a Q| + EPS on the value Q
NORMALIZE = True
# gap of -BETA continues at 1/2; larger is softer
BETA = 0.0
# shift subtracted from the signed gap before the cap at 0
BIAS = 0.1
# floor on |max_a Q|; required when NORMALIZE is True
EPS = 1e-06

# Optimizer learning rates
# token embeddings and numeric Fourier features
LR_EMBEDDING = 1e-05
# backbone
LR_BACKBONE = 1e-05
# prediction heads and their input RMSNorms
LR_HEADS = 1e-05

# Target interpolation rates
# delayed embedding
POLYAK_TAU_EMBEDDING = 0.1
# delayed backbone
POLYAK_TAU_BACKBONE = 0.1
# delayed value head
POLYAK_TAU_VALUE = 0.001

# Runtime
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Payload names are selected explicitly; the underscore is only a convention.
environment_episode_ranges = partial(
    environment_episode_ranges,
    protocol=EnvironmentProtocol(
        episode_start="_episode_start",
        terminated="_terminated",
        truncated="_truncated",
    ),
)

## Load Data

`load_stores_from_hub` downloads the dataset snapshot and reconstructs the saved
`Datastore` objects. Each returned store is one ordered environment stream.


In [ ]:
stores = load_stores_from_hub(repo_id=DATASET_ID, split="train", force_download=True)

## Data Pipeline

Use the event pipeline described in [02 — Offline DQN](02_train_offline_dqn.ipynb):
`source events → sparse decisions → augmenter → tokenizer → pack → embedding`.

Compose `train_transform = compose(stages=(augmenter.events, tokenizer.encode))` and
pass `train_transform` to the loader. The augmenter reads event payloads directly. Each
sampled outer task gets a separate `group_id`, retaining its repeated lake episodes
together while isolating attention and objective transitions from other tasks.
`TOKEN_BUDGET` counts packed tokens, including episode prompts; the loader keeps whole
samples and rejects an oversized first sample.

For evaluation, adapt [example 15](15_inference.ipynb) with this checkpoint's model
settings and saved tokenizer. Omit training augmentation so selected actions match the
environment.


In [ ]:
def episode_separator(ctx):
    ended = ctx.get("inner_terminated", False) or ctx.get("inner_truncated", False)
    return "\n\n" if ended else ""


# Pipeline order: augmenter → tokenizer
episode_prompt = (
    "Your job is to predict the future sum of rewards in FrozenLake. "
    "Navigate a grid; reach the goal for reward; a hole ends the "
    "episode with none. Each episode lasts at most "
    f"{MAX_STEPS_PER_EPISODE} steps. You have {REPEAT_COUNT} "
    "episodes to solve the task. The grid is permuted, so squares are "
    "not in order; action ids may be remapped, then rendered as a, b, c, d.\n"
    "Strategy: explore; keep a mental map of what has and has not been "
    "explored; avoid holes you have already fallen in; once you have a "
    "path to the goal, repeat it.\n"
    "At each action-request newline, predict the value of all actions. Format:\n"
    "action letter,observation[,r=reward][,terminated][,truncated].\n"
)

augmenter = Augmenter(
    seed_field="episode_index",
    fields=[
        {
            "type": "discrete_permute",
            "provider": None,
            "input_field": "action",
            "input_vector_field": "_q_star",
            "vocab_size": MAX_ACTIONS,
        },
        {
            "type": "discrete_permute",
            "provider": None,
            "input_field": "observation",
            "vocab_size": MAX_OBS_DISCRETE,
        },
    ],
)

tokenizer = Tokenizer(
    tokenizers={"text": PRETRAINED},
    step_field="step_index",
    input_fields=[
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "coordinator",
            "input_field": "_episode_start",
            "format": episode_prompt,
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "repeat_episode_index",
            "format": "episode {field+1}\n",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "agent",
            "input_field": "action",
            "lookup": ["a", "b", "c", "d"],
            "format": "{field}",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "observation",
            "format": ",{field}",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "reward",
            "format": ",r={field:g}",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "inner_terminated",
            "format": ",terminated",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "inner_truncated",
            "format": ",truncated",
        },
        {
            "type": "token",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "action_requested",
            "format": "\n",
            "head_output": True,
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "format": episode_separator,
        },
    ],
    objective_fields=[
        {
            "input_field": "action",
            "missing_value": None,
        },
        {
            "input_field": "reward",
            "missing_value": 0.0,
        },
        {
            "input_field": "_terminated",
            "output_field": "terminated",
            "missing_value": False,
        },
        {
            "input_field": "_truncated",
            "output_field": "truncated",
            "missing_value": False,
        },
    ],
)

train_transform = compose(stages=(augmenter.events, tokenizer.encode))


loader = DataLoader(
    stores=stores,
    token_budget=TOKEN_BUDGET,
    sample_ranges=environment_episode_ranges,
    transform=train_transform,
    prefetch=4,
    num_workers=4,
)

## Build Encoder and Heads

Call the encoder once, then apply the head(s) to its encodings. Cached inference uses
the same encoder and returns cache and readout-mask metadata. A `ModuleDict` called
`modules` holds references to the online components for checkpointing and mode changes;
it is not called for predictions.

The delayed encoder is a frozen copy. Separate backup value/action heads track the
online Q head(s) with independent Polyak rates. Only online parameters enter the
optimizer.


In [ ]:
class Encoder(Model):
    def __init__(
        self, *, pretrained, tokenizer_sources, train_kernel, decode_kernel, dtype
    ):
        super().__init__()
        self.embedding = MultiEmbedding(
            embeddings={
                name: Embedding(pretrained=source)
                for name, source in tokenizer_sources.items()
            }
        )
        self.backbone = Backbone(
            pretrained=pretrained,
            train_kernel=train_kernel,
            decode_kernel=decode_kernel,
            dtype=dtype,
        )

    def forward(self, tokens, *, cache=None, use_cache=False):
        if cache is not None and not use_cache:
            raise ValueError("Passing cache= requires use_cache=True.")
        tensors = tokens.to(next(self.parameters()).device)
        token_ids = tensors["token_ids"]
        numeric_values = tensors["numeric_values"]
        numeric_mask = tensors["numeric_mask"]
        sequence_ids = tensors["sequence_ids"]
        head_token_ids = tensors["head_token_mask"].nonzero(as_tuple=True)[0]
        embed = self.embedding(
            token_ids,
            numeric_values=numeric_values,
            numeric_mask=numeric_mask,
            numeric_field_ids=tensors["numeric_field_ids"],
            numeric_field_names=tokens["numeric_field_names"],
            tokenizer_ids=tensors["tokenizer_ids"],
            tokenizer_names=tokens["tokenizer_names"],
        )
        if use_cache:
            embed, lengths, head_token_ids, _ = prepare_decode(
                embeds=embed,
                tokens=tokens,
                sequence_ids=sequence_ids,
                head_token_mask=tensors["head_token_mask"],
            )
            if cache is None:
                cache = DecodeCache(
                    sessions=(
                        self.backbone.decode_session(
                            batch_size=tokens["num_sequences"]
                        ),
                    )
                )
            if len(cache.sessions) != 1:
                raise ValueError("expected one backbone cache session")
            features = self.backbone(embed, cache=cache.sessions[0], lengths=lengths)
            head_gather_ids = head_token_ids.unsqueeze(-1).expand(
                *head_token_ids.shape,
                features.shape[-1],
            )
            h = features.gather(1, head_gather_ids)
        else:
            features = self.backbone(embed, sequence_ids=sequence_ids)
            h = features[head_token_ids]
        return {
            "z": h,
            "cache": cache,
        }


class QHead(nn.Module):
    def __init__(self, *, hidden_dim, num_actions, num_layers, scale):
        super().__init__()
        self.mlp = MLP(
            in_features=hidden_dim,
            out_features=num_actions,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            scale=scale,
            propagate_gradient=1.0,
        )

    def forward(self, z):
        return self.mlp(z)


encoder = (
    Encoder(
        tokenizer_sources=tokenizer.tokenizer_sources,
        pretrained=PRETRAINED,
        train_kernel="flex",
        decode_kernel="flex",
        dtype=torch.float32,
    )
    .train()
    .to(device)
)

q_head = (
    QHead(
        hidden_dim=encoder.backbone.hidden_dim,
        num_actions=MAX_ACTIONS,
        num_layers=HEAD_LAYERS,
        scale=HEAD_SCALE,
    )
    .train()
    .to(device)
)

# A checkpoint container; forward calls use the encoder and heads separately.
modules = nn.ModuleDict({"encoder": encoder, "q_head": q_head})

delayed_encoder = deepcopy(encoder).requires_grad_(False).eval()
backup_value_head = deepcopy(q_head).requires_grad_(False).eval()

print(encoder)
print(q_head)

## Train

Each outer cycle runs `TRAIN_STEPS` optimizer updates via `run_train`. Mouse Core
abstractions do most of the work:

`value_gap_gate(beta=BETA, normalize=NORMALIZE, policy_delayed=False, value_delayed=False, bias=BIAS, eps=EPS)`
is the value-gap continuation when `NORMALIZE` is `True`. `normalize=False` is
`value_gap_gate(beta=BETA, normalize=False, policy_delayed=False, value_delayed=False, bias=BIAS)`,
with no `eps`. `a*` and the taken action are read from the current head. `bias` is
subtracted from the signed gap, then the gap is capped at `0`. Along a run

`G_t = r_{t+1} + γ_{t+1} ((1 - c_{t+1}) V(s_{t+1}) + c_{t+1} G_{t+1})`

with `gap_s = min(0, (Q(s, a_s) - V) / (|max_a Q| + EPS) - BIAS)` when `NORMALIZE` is
`True`, and `c_s = 2 ** (gap_s / BETA)`, on the value Q after `value`. `V = Q(s, a*)`
and `a*` is the argmax of the current head. `NORMALIZE = False` drops the denominator
and uses `gap_s = min(0, Q(s, a_s) - V - BIAS)`, and that call omits `eps`. `|max_a Q|`
is the absolute value of the largest action value of the value Q at `s`. `V` in the
backup comes from the separate delayed heads: the backup action scores choose the action
and the backup value scores evaluate it. Positive temperature uses the action-score
policy's expected backup value plus entropy. A zero gap at `s_{t+1}` carries `G_{t+1}`
in full (`c = 1`). A gap of `-BETA` continues at `1/2`. A larger `BETA` is a softer cut.
A negative gap drops that term and bootstraps `V(s_{t+1})`.

`boundary_discount` ends the trace at outer task termination or truncation. The reward
and value transforms are disabled explicitly with `None`; `temperature=0.0` selects the
highest-scoring backup action.

`environment_episode_ranges` selects complete outer tasks from the event log. Unfinished
tails remain in storage. The objective raises if a training target needs a value beyond
its sampled group; zero discount at outer termination or truncation ends the return.

The loader returns `inputs` (tokens, embedding features, sequence IDs, and
`head_token_mask`) and `objective_data` (training labels and step mappings).
`objective_data["head_step_ids"][p]` tells the loss which step owns prediction row `p`,
including steps with zero or multiple predictions. Pass this existing mapping to the
objective; no separate prediction-count column is needed. An absent action is
represented by `NaN` in the packed labels.


In [ ]:
optimizer = AdamW(
    params=[
        *encoder.parameter_groups(
            lr={"embedding": LR_EMBEDDING, "backbone": LR_BACKBONE}
        ),
        {"params": q_head.parameters(), "lr": LR_HEADS},
    ],
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1e-08,
)
objective = DqnObjective(
    reward=None,
    value=None,
    discount=boundary_discount(
        gamma_step=1.0,
        gamma_terminated=0.0,
        gamma_truncated=0.0,
    ),
    temperature=0.0,
    gate=(
        value_gap_gate(
            beta=BETA,
            normalize=True,
            policy_delayed=False,
            value_delayed=False,
            bias=BIAS,
            eps=EPS,
        )
        if NORMALIZE
        else value_gap_gate(
            beta=BETA,
            normalize=False,
            policy_delayed=False,
            value_delayed=False,
            bias=BIAS,
        )
    ),
)


def run_train(
    *, num_steps: int
) -> tuple[torch.Tensor, dict[str, float | torch.Tensor]]:
    """Run ``num_steps`` optimizer steps on batches from ``loader``."""
    modules.train()
    loss: torch.Tensor | None = None
    metrics: dict[str, float | torch.Tensor] = {}
    for _ in range(num_steps):
        inputs, objective_data = loader.next_batch()
        inputs = inputs.to(device)
        objective_data = objective_data.to(device)
        group_id = objective_data["step_sequence_ids"]
        out = encoder(inputs)
        with torch.no_grad():
            delayed_out = delayed_encoder(inputs)
            backup_action = q_head(delayed_out["z"])
        losses, metrics = objective(
            objective_data=objective_data.to(device),
            group_id=group_id.to(device),
            head_step_ids=objective_data["head_step_ids"],
            predictions=q_head(out["z"]),
            backup_value_predictions=backup_value_head(delayed_out["z"]),
            backup_action_predictions=backup_action,
        )
        loss = torch.stack(tuple(losses.values())).sum()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        polyak_update(
            target=delayed_encoder.embedding,
            source=encoder.embedding,
            tau=POLYAK_TAU_EMBEDDING,
        )
        polyak_update(
            target=delayed_encoder.backbone,
            source=encoder.backbone,
            tau=POLYAK_TAU_BACKBONE,
        )
        polyak_update(target=backup_value_head, source=q_head, tau=POLYAK_TAU_VALUE)
    assert loss is not None
    return (loss, metrics)

## Run

Each of `NUM_CYCLES` cycles calls `run_train(num_steps=TRAIN_STEPS)`. Score the
checkpoint later in `15_inference.ipynb`.


In [ ]:
for cycle in range(NUM_CYCLES):
    loss, metrics = run_train(num_steps=TRAIN_STEPS)
    print(
        f"cycle={cycle} train  loss={loss.item():.4f}  q={metrics['q_values_mean']:.3f}"
    )

loader.close()

## Save To The Hub

The checkpoint contains the encoder and separate head(s) in `modules`. The model weights
and tokenizer packing specification share `MODEL_ID`. Reconstruct the same component
structure before loading. Checkpoints from the old architecture with a projection are
not compatible with this example.


In [ ]:
modules.eval().to("cpu")
model_url = push_model_to_hub(
    model=modules,
    tokenizer=tokenizer,
    repo_id=MODEL_ID,
    private=False,
    clear=True,
)
print(f"Pushed encoder, heads, and tokenizer to {model_url}")